# 201 · Dynamic vs IDL binary

Companion to [Dynamic vs IDL binary](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/dynamic-vs-idl-binary/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/201/dynamic_vs_idl_binary.ipynb)

After rejecting text JSON: flexible documents (MessagePack-class) vs long-lived product contract (IDL/field numbers). Choose by **contract investment**, not a single latency chart.

> **Honesty:** sizes illustrative for this record only.


In [ ]:
import json
import struct

RECORD = {
    "order_id": 1001,
    "sku": "ABC-42",
    "qty": 3,
    "price_cents": 1999,
}

try:
    import msgpack
    HAS_MSGPACK = True
except ImportError:
    HAS_MSGPACK = False
    print("pip install msgpack recommended for this notebook")



In [ ]:
def encode_varint(u: int) -> bytes:
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


def encode_key(fn: int, wt: int) -> bytes:
    return encode_varint((fn << 3) | wt)


def encode_idl(r: dict) -> bytes:
    # 1 order_id, 2 sku, 3 qty, 4 price_cents — all varint/string
    out = bytearray()
    out += encode_key(1, 0) + encode_varint(r["order_id"])
    b = r["sku"].encode()
    out += encode_key(2, 2) + encode_varint(len(b)) + b
    out += encode_key(3, 0) + encode_varint(r["qty"])
    out += encode_key(4, 0) + encode_varint(r["price_cents"])
    return bytes(out)


j = json.dumps(RECORD, separators=(",", ":")).encode()
idl = encode_idl(RECORD)
rows = [("JSON (text baseline)", len(j)), ("IDL binary sketch", len(idl))]
if HAS_MSGPACK:
    mp_map = msgpack.packb(RECORD, use_bin_type=True)
    mp_arr = msgpack.packb(
        [RECORD["order_id"], RECORD["sku"], RECORD["qty"], RECORD["price_cents"]],
        use_bin_type=True,
    )
    rows[1:1] = [
        ("MessagePack map", len(mp_map)),
        ("MessagePack array", len(mp_arr)),
    ]
print(f"{'encoding':24} {'bytes':>6}")
for name, n in rows:
    print(f"{name:24} {n:6}")
print("IDL hex:", idl.hex(" "))



## Decision frame (from the article)

| Prefer dynamic binary when… | Prefer IDL binary when… |
|-----------------------------|-------------------------|
| Document shapes vary | Multi-year product interface |
| No IDL toolchain | Multi-language stubs + field-number discipline |
| Boundary validation is enough | Explicit compatibility rules |

Neither replaces JSON for public human-debuggable APIs without a plan.


## Flexibility demo

Add `note`: dynamic map/JSON absorbs it; the fixed IDL sketch has no field until you allocate a number.


In [ ]:
flexible = dict(RECORD)
flexible["note"] = "rush"
if HAS_MSGPACK:
    print("msgpack with extra field nbytes", len(msgpack.packb(flexible, use_bin_type=True)))
print("JSON with extra field:", json.dumps(flexible, separators=(",", ":")))
print("IDL sketch above has no slot for 'note' until you allocate field 5+ in the schema.")



## Takeaways

Dynamic ≈ JSON model + binary tags. IDL ≈ schema + field numbers + codegen.

**Next:** [Compression vs format](./compression_vs_format.ipynb)
